# Creditworthiness Modelling: XGBClassifiers with native categorical treatment

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, roc_auc_score, f1_score, 
                            classification_report, confusion_matrix, ConfusionMatrixDisplay, 
                            precision_recall_curve, PrecisionRecallDisplay)

## <font color="orange"> Data loading <font>

In [82]:
# load data
path = "../data/processed-data/processed_data.csv"
df = pd.read_csv(path)

In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32574 entries, 0 to 32573
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   person_age_years                  32574 non-null  int64  
 1   person_home_ownership             32574 non-null  str    
 2   person_emp_length_years           31679 non-null  float64
 3   loan_intent                       32574 non-null  str    
 4   loan_grade                        32574 non-null  str    
 5   loan_int_rate                     29459 non-null  float64
 6   loan_status                       32574 non-null  int64  
 7   cb_person_default_on_file         32574 non-null  int64  
 8   cb_person_cred_hist_length_years  32574 non-null  int64  
 9   loan_term_months                  32574 non-null  int64  
 10  loan_to_income_ratio              32574 non-null  float64
 11  open_accounts                     32574 non-null  int64  
 12  credit_utilizat

In [84]:
# X/y split
y = df.pop("loan_status")
X = df.copy()

In [85]:
# specify categorical
ohe_features = X.select_dtypes(exclude=np.number).columns.tolist()
X[ohe_features] = X[ohe_features].astype("category")
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 32574 entries, 0 to 32573
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   person_age_years                  32574 non-null  int64   
 1   person_home_ownership             32574 non-null  category
 2   person_emp_length_years           31679 non-null  float64 
 3   loan_intent                       32574 non-null  category
 4   loan_grade                        32574 non-null  category
 5   loan_int_rate                     29459 non-null  float64 
 6   cb_person_default_on_file         32574 non-null  int64   
 7   cb_person_cred_hist_length_years  32574 non-null  int64   
 8   loan_term_months                  32574 non-null  int64   
 9   loan_to_income_ratio              32574 non-null  float64 
 10  open_accounts                     32574 non-null  int64   
 11  credit_utilization_ratio          32574 non-null  float64 
 12  p

In [86]:
# train-test stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## <font color="orange"> XGBClassifier with native categorical treatment <font>

In [87]:
# cv 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# best Fbeta score
BETA = 1.0   # 1.0 for F1, 2.0 for F2
def best_fbeta(y_true, p, beta=BETA):
    prec, rec, thr = precision_recall_curve(y_true, p)
    b2 = beta ** 2
    f = (1 + b2) * prec * rec / (b2 * prec + rec + 1e-12)
    i = np.argmax(f[:-1])          # last point has no threshold
    return f[i], thr[i]

In [88]:
y_arr = np.asarray(y_train)
# validation splitting
folds = []
for trn, val in cv.split(X_train, y_arr):
    folds.append((
        X_train.iloc[trn], y_arr[trn],
        X_train.iloc[val], y_arr[val],
    ))

def params_set(trial):
    return dict(
        learning_rate=trial.suggest_float("eta", 5e-3, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 4, 7),
        min_child_weight=trial.suggest_float("min_child_weight", 2.0, 20.0, log=True),
        subsample=trial.suggest_float("subsample", 0.4, 0.95),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 0.95),
        gamma=trial.suggest_float("gamma", 0.5, 20, log=True),
        #reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 2, 20, log=True),
        max_cat_threshold = trial.suggest_int("max_cat_threshold", 1, len(ohe_features))
        #scale_pos_weight=trial.suggest_float("scale_pos_weight", 0.5*ib_ratio, ib_ratio),
    )

def objective(trial):
    params = params_set(trial)
    #scores, best_iters = [], []
    scores, thresholds, best_iters = [], [], []
    for step, (Xtr, ytr, Xva, yva) in enumerate(folds):
        model = XGBClassifier(
            **params,
            n_estimators=850,
            early_stopping_rounds=50,
            eval_metric="aucpr",
            tree_method="hist",
            objective="binary:logistic",
            enable_categorical=True,
            max_cat_to_onehot = 4,
            n_jobs=-1,        
            random_state=42,
        )
        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        #scores.append(average_precision_score(yva, model.predict_proba(Xva)[:, 1]))
        p = model.predict_proba(Xva)[:, 1]
        f, t = best_fbeta(yva, p)
        scores.append(f)
        thresholds.append(t)
        best_iters.append(model.best_iteration + 1)

        trial.report(np.mean(scores), step)
        # prune bad trials
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("n_estimators", int(np.mean(best_iters)))
    return float(np.mean(scores))

def optuna_study(ntrials=100):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=2),
    )
    study.optimize(objective, n_trials=ntrials)
    return study

In [89]:
# out-of-fold threshold
def oof_threshold(best_params, n_estimators, folds, beta=BETA):
    oof_p, oof_y = [], []
    # retrain on folds
    for Xtr, ytr, Xva, yva in folds:
        m = XGBClassifier(**best_params, n_estimators=n_estimators, enable_categorical=True, max_cat_to_onehot = 4,
                          tree_method="hist", objective="binary:logistic", n_jobs=-1, random_state=42).fit(Xtr, ytr)
        oof_p.append(m.predict_proba(Xva)[:, 1]) 
        oof_y.append(yva)
    #p, y = np.concatenate(oof_p), np.concatenate(oof_y)
    #prec, rec, thr = precision_recall_curve(y, p)
    #f1 = 2 * prec * rec / (prec + rec + 1e-12)
    # threshold at max f1
    #i = np.argmax(f1[:-1])
    return best_fbeta(np.concatenate(oof_y), np.concatenate(oof_p), beta)[::-1] # thr[i], f1[i]

In [90]:
def report(name, pipeline, X, y, threshold):
    """Print metrics and show a confusion matrix for one dataset."""
    prob = pipeline.predict_proba(X)[:, 1]
    pred = (prob >= threshold).astype(int)

    print(f"\n{name} F1:                {f1_score(y, pred):.4f}")
    print(f"{name} balanced accuracy: {balanced_accuracy_score(y, pred):.4f}")
    print(f"{name} ROC-AUC:           {roc_auc_score(y, prob):.4f}")
    print(f"{name} average precision: {average_precision_score(y, prob):.4f}")
    print(f"\n{name} classification report:")
    print(classification_report(y, pred, zero_division=0, digits=3))

    ConfusionMatrixDisplay.from_predictions(y, pred)
    plt.title(f"{name} (threshold = {threshold:.3f})")
    plt.show()


In [91]:
def retrain_and_evaluate(study, folds, X_train, y_train, X_test, y_test,
                         n_estimators_scale=1.1):
    best = study.best_trial
    params = dict(best.params)
    params["learning_rate"] = params.pop("eta")

    # Number of trees found by early stopping, scaled up slightly because the
    # final fit sees about 25% more data than each CV training split
    n_estimators = int(best.user_attrs["n_estimators"] * n_estimators_scale)

    threshold, oof_f1 = oof_threshold(params, best.user_attrs["n_estimators"], folds)

    pipeline = Pipeline([
        ("classifier", XGBClassifier(
            **params,
            n_estimators=n_estimators,
            tree_method="hist",
            objective="binary:logistic",
            enable_categorical=True,
            max_cat_to_onehot = 4,
            n_jobs=-1,
            random_state=42,
        )),
    ])
    pipeline.fit(X_train, y_train)

    print(f"Best CV f1: {study.best_value:.4f}")
    print(f"OOF F1 at tuned threshold: {oof_f1:.4f}")
    print(f"Threshold:                 {threshold:.3f}")
    print(f"n_estimators:              {n_estimators}")

    report("Test", pipeline, X_test, y_test, threshold)
    report("Training", pipeline, X_train, y_train, threshold)

    return pipeline, threshold

In [92]:
study = optuna_study(ntrials=120)
pipeline, threshold = retrain_and_evaluate(
    study, folds, X_train, y_train, X_test, y_test
)

[I 2026-11-23 01:46:49,686] A new study created in memory with name: no-name-f61cab77-88cb-4a6f-bfba-1b1ae4553928
[I 2026-11-23 01:47:26,940] Trial 0 finished with value: 0.8408425707434631 and parameters: {'eta': 0.015355286838886862, 'max_depth': 7, 'min_child_weight': 10.790061933340453, 'subsample': 0.72926216630837, 'colsample_bytree': 0.48581025224334007, 'gamma': 0.8889607461211928, 'reg_lambda': 2.2861967752626438, 'max_cat_threshold': 3}. Best is trial 0 with value: 0.8408425707434631.
[I 2026-11-23 01:47:47,597] Trial 1 finished with value: 0.7798994924861212 and parameters: {'eta': 0.03027182927734624, 'max_depth': 6, 'min_child_weight': 2.0970775450389234, 'subsample': 0.9334504186890968, 'colsample_bytree': 0.8578434524402319, 'gamma': 1.094334266006264, 'reg_lambda': 3.0398696602619606, 'max_cat_threshold': 1}. Best is trial 0 with value: 0.8408425707434631.
[I 2026-11-23 01:48:19,129] Trial 2 finished with value: 0.8405475814563212 and parameters: {'eta': 0.0124393672099

KeyboardInterrupt: 

In [ ]:
# params importance
optuna.importance.get_param_importances(study)

{'colsample_bytree': 0.32020286210354676,
 'min_child_weight': 0.1898061218955653,
 'eta': 0.15835576920196734,
 'subsample': 0.1264792963946588,
 'gamma': 0.08335302052561284,
 'reg_lambda': 0.07104048969752484,
 'max_depth': 0.04477070053733836,
 'max_cat_threshold': 0.005991739643785749}

In [ ]:
df = study.trials_dataframe()
top = df.sort_values("value", ascending=False).head(15)
print(top[["value", "params_max_depth", "params_min_child_weight", "params_gamma", "params_subsample", "params_colsample_bytree", "params_reg_lambda"]])
df.groupby("params_max_depth")["value"].agg(["max", "mean", "count"])

        value  params_max_depth  params_min_child_weight  params_gamma  \
110  0.848207                 6                 2.286970      0.679629   
94   0.848172                 6                 2.195304      0.732603   
107  0.847982                 6                 2.081808      0.801485   
104  0.847900                 6                 2.319121      0.702050   
101  0.847852                 6                 2.519645      0.646598   
86   0.847795                 6                 2.190471      0.642174   
111  0.847757                 6                 2.258677      1.048672   
116  0.847726                 6                 2.184910      0.524483   
72   0.847713                 6                 2.835932      0.872009   
75   0.847624                 6                 2.163112      0.971646   
93   0.847606                 6                 2.079911      0.794156   
108  0.847596                 6                 2.439949      0.650988   
103  0.847595                 6       

,max,mean,count
params_max_depth,,,
4,0.823279,0.820264,2
5,0.843797,0.840175,2
6,0.848207,0.844803,87
7,0.845735,0.834325,29
